# Measure training throughput and peak GPU memory

Produces the two numbers marked **not yet measured** in `docs/RESULTS.md`:
average training throughput in tokens/sec, and peak GPU memory in GB.

Self-contained: it carries the project's source files inline, so it needs no
repository, no dataset upload and no setup. It runs at the **exact shape of
the original training run** (32 x 128, gradient accumulation 32, lr 4e-4,
fp16), because throughput at a different batch shape is a different number.

## Before pressing Run All

1. **Session options > Accelerator > GPU P100**
2. **Session options > Internet > On**

### Read this if you are on a P100

Kaggle's preinstalled PyTorch is compiled for compute capability 7.0 and
above. The P100 is 6.0, so the stock build reports `cuda.is_available() ==
True`, moves tensors to the GPU, and then fails on the first kernel launch
with `no kernel image is available for execution on the device`.

Section 2 detects this and installs a PyTorch build that still ships Pascal
kernels. Section 3 then proves the GPU can actually run one **before** any
time is spent tokenizing.

Total runtime is roughly 20 minutes, including the PyTorch reinstall.


## 1. What hardware did Kaggle give us

Throughput is a property of code *and* hardware, so this has to be recorded
alongside any number produced here.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch

assert torch.cuda.is_available(), 'No GPU. Set Accelerator to GPU and restart the session.'
GPU_NAME = torch.cuda.get_device_name(0)
MAJOR, MINOR = torch.cuda.get_device_capability(0)
IS_PASCAL = (MAJOR, MINOR) < (7, 0)

print(f'torch      : {torch.__version__}')
print(f'gpu        : {GPU_NAME}')
print(f'capability : sm_{MAJOR}{MINOR}')
print(f'pascal fix : {"REQUIRED" if IS_PASCAL else "not needed"}')

if 'P100' not in GPU_NAME:
    print(f'\nNOTE: this is a {GPU_NAME}, not a P100.')
    print('The measurement will be valid, but label it with THIS gpu name,')
    print('not P100, when you report it.')


## 2. Install a Pascal-capable PyTorch, only if needed

PyTorch stopped shipping `sm_60` kernels in its recent CUDA 12.6+ builds. The
2.5.1 + cu121 wheel still contains them.

**No kernel restart is needed.** Training runs as a subprocess
(`!python train.py`), which starts a fresh interpreter and picks up the newly
installed package automatically. Only the notebook's own already-imported
`torch` stays stale, which is why every check below runs in a subprocess too.

This cell does nothing on a T4, A100 or any other sm_70+ GPU.


In [ ]:
if IS_PASCAL:
    print(f'{GPU_NAME} is sm_{MAJOR}{MINOR}; the stock torch has no kernels for it.')
    print('Installing torch 2.5.1 (cu121), which still ships Pascal kernels.')
    print('This takes a few minutes.\n')
    !pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
    print('\ndone')
else:
    print(f'{GPU_NAME} is sm_{MAJOR}{MINOR}; the preinstalled torch is fine.')


In [ ]:
# tiktoken (GPT-2 BPE) and datasets (FineWeb-Edu) are not on the Kaggle image.
# torch, numpy, tqdm and matplotlib are.
!pip install -q tiktoken datasets


## 3. Prove the GPU can actually run a kernel

This is the check whose absence wasted a tokenization pass. `is_available()`
being True says nothing about whether the installed binary has kernels for
this chip. The only way to know is to launch one.

It runs in a subprocess so it sees whatever torch section 2 installed.

**If this fails, stop.** Nothing below will work. Switch the accelerator to
T4 x2 and re-run from the top; a T4 measurement is still a real measurement,
it just has to be labelled T4.


In [ ]:
%%bash
python - <<'PY'
import sys, torch
print('torch     :', torch.__version__)
print('gpu       :', torch.cuda.get_device_name(0))
print('capability: sm_%d%d' % torch.cuda.get_device_capability(0))
try:
    a = torch.randn(512, 512, device='cuda')
    b = (a @ a).sum().item()          # forces a real kernel launch
    torch.cuda.synchronize()
    h = torch.nn.Linear(64, 64).cuda()(torch.randn(8, 64, device='cuda')).sum()
    h.backward()                       # and a real backward kernel
    torch.cuda.synchronize()
    print('\nCUDA SMOKE TEST PASSED (matmul checksum %.3f)' % b)
except Exception as e:
    print('\nCUDA SMOKE TEST FAILED:', type(e).__name__)
    print(e)
    print('\nDo not continue. Either the Pascal fix in section 2 did not apply,')
    print('or this GPU is unsupported. Switch Accelerator to T4 x2 and rerun.')
    sys.exit(1)
PY


## 4. Write the project source files

Each cell writes one file exactly as it appears in the repository. Section 8
prints their hashes so you can confirm nothing was altered.


In [ ]:
%%writefile config.py
"""Model and training configuration.

Every value here is the one that produced the released checkpoint, except where
a comment says otherwise. Keeping the configuration in one importable place is
the single change that would have prevented the most serious bug in the original
training run: the notebook defined `batch_size` and `block_size` twice, in two
different cells, and the batch sampler silently read the second pair. See
docs/AUDIT.md.
"""

# Architecture of the released checkpoint.
#
# Naming note: the original notebook called this dict GPT_CONFIG_124M because it
# was modelled on GPT-2-small. The actual parameter count is 134,077,440, since
# this model has 8 layers rather than GPT-2-small's 12 and does not tie the
# output head to the input embedding. It is renamed here to match reality;
# tests/test_model.py asserts the exact count.
GPT_CONFIG_134M = {
    "vocab_size": 50257,      # GPT-2 BPE vocabulary (tiktoken "gpt2")
    "context_length": 256,    # positions the model allocates embeddings for
    "emb_dim": 768,           # model width, d_model
    "n_heads": 12,            # 768 / 12 = 64 dimensions per head
    "n_layers": 8,            # transformer blocks
    "drop_rate": 0.1,         # dropout on embeddings, attention and residuals
    "qkv_bias": False,        # no bias on the Q/K/V projections
}


# Hyperparameters of the run that produced the released checkpoint.
#
# These are the values that actually reached the model. The original notebook
# also contained a second, unused set (peak lr 1e-4, betas 0.9/0.95, a
# warmup-plus-cosine schedule) that was attached to an optimizer the training
# cell then replaced, so it never affected training. docs/AUDIT.md explains it.
TRAIN_CONFIG = {
    "batch_size": 32,                    # sequences per micro-batch
    "block_size": 128,                   # tokens per sequence, the REAL context used
    "gradient_accumulation_steps": 32,   # micro-batches per optimizer step
    "num_batches_per_epoch": 1000,       # micro-batches in one "epoch" of random windows
    "num_epochs": 1,                     # the training cell was re-run repeatedly
    "learning_rate": 4e-4,               # flat: see AUDIT.md, the schedule never applied
    "weight_decay": 0.1,
    "grad_clip": 1.0,
    "eval_freq": 100,                    # optimizer steps between evaluations
    "eval_iter": 10,                     # batches averaged per loss estimate
    "precision_dtype": "float16",        # mixed precision via torch.amp
}

# The learning rate above is FIXED. No schedule was applied to the released
# checkpoint: the notebook built one but bound it to an optimizer the training
# cell then replaced, so it never reached the weights (docs/AUDIT.md, bug 2).
# train.py implements warmup and cosine decay correctly for future runs.

# Hardware the released checkpoint was trained on. Recorded here because
# throughput and memory figures are meaningless without it, and because the
# original run logged nothing at all.
HARDWARE = {
    "gpu": "NVIDIA Tesla P100-PCIE-16GB",
    "provider": "Kaggle, free tier",
    "notes": "P100 supports full-rate fp16 but not bfloat16, and its compute "
             "capability 6.0 is below the 7.0 that torch.compile's Triton "
             "backend requires. Hence --dtype float16 and no compilation.",
}


# Where the tokenized corpora live. These are uint16 memory-mapped token files
# produced by tokenize_data.py, not text.
DATA_CONFIG = {
    "train_bin": "train.bin",
    "val_bin": "validation.bin",
    "dataset": "HuggingFaceFW/fineweb-edu",
    "dump": "CC-MAIN-2024-10",   # the Common Crawl snapshot used for training
    "max_tokens": 8_000_000_000,  # size of the corpus written to disk
}


def describe(cfg=GPT_CONFIG_134M):
    """Human-readable one-liner, used by the scripts when they start up."""
    return (f"{cfg['n_layers']}L / {cfg['n_heads']}H / {cfg['emb_dim']}d, "
            f"context {cfg['context_length']}, vocab {cfg['vocab_size']}")


In [ ]:
%%writefile model.py
"""The 134M-parameter GPT, written from scratch in PyTorch.

Nothing here is imported from a transformer library. Attention, the feed-forward
block, layer normalization and the causal mask are all written out explicitly,
because the point of the project was to understand them rather than to call them.

Structure, bottom up:

    LayerNorm            normalize each token's feature vector, then rescale
    FeedForward          two linear layers with a ReLU between them, 4x wide
    MultiHeadAttention   causal self-attention over `n_heads` parallel heads
    TransformerBlock     pre-norm attention + pre-norm feed-forward, both residual
    GPTModel             embeddings -> N blocks -> final norm -> vocabulary logits

Architecture is verified against the released checkpoint by
tests/test_model.py, which asserts the exact parameter count of 134,077,440.
"""
import torch
import torch.nn as nn


class LayerNorm(nn.Module):
    """Layer normalization, written out rather than using nn.LayerNorm.

    For each token independently: subtract the mean of its 768 features, divide
    by their standard deviation, then apply a learned per-feature scale and
    shift. This keeps activations in a stable range as they pass through 8
    blocks, which is what makes deep stacks trainable at all.

    `unbiased=False` divides the variance by N rather than N-1, matching GPT-2.
    """

    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5  # guards against division by zero
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class FeedForward(nn.Module):
    """Position-wise feed-forward network: 768 -> 3072 -> ReLU -> 768.

    Applied to each token independently. The 4x widening is the standard
    transformer ratio, and this block holds roughly two thirds of the model's
    non-embedding parameters.
    """

    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            nn.ReLU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class MultiHeadAttention(nn.Module):
    """Causal multi-head self-attention.

    Each token builds a query, a key and a value. Attention scores are the dot
    products of queries against keys, scaled by 1/sqrt(head_dim) so the softmax
    does not saturate as head_dim grows. The causal mask sets every score for a
    future position to -inf, so after the softmax those positions carry zero
    weight and a token can only attend to itself and its past. That masking is
    what makes next-token prediction a valid training objective: without it the
    model could read the answer.

    The 768 dimensions are split into 12 heads of 64. The heads are computed in
    parallel as a single batched matmul by reshaping to
    (batch, heads, tokens, head_dim), not by looping.
    """

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # mixes the heads back together
        self.dropout = nn.Dropout(dropout)

        # Upper-triangular matrix of ones, excluding the diagonal. Registered as
        # a buffer so it moves with .to(device) and is saved in the checkpoint,
        # but receives no gradient.
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)        # (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Split the last dimension into heads:
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Put the head dimension next to the batch so every head is one matmul:
        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # (b, num_heads, num_tokens, num_tokens): how much each token attends to
        # each other token, per head
        attn_scores = queries @ keys.transpose(2, 3)

        # Crop the mask to this sequence length so shorter inputs still work
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Weighted sum of values, then put the heads back on the last dimension
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)


class TransformerBlock(nn.Module):
    """One transformer block: attention, then feed-forward, both residual.

    Pre-norm: normalization happens *before* each sublayer rather than after, so
    the residual path from input to output is never normalized. That keeps
    gradients well behaved through a deep stack and is why this trains without a
    carefully tuned warmup, unlike the original post-norm transformer.
    """

    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Attention sublayer
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # Feed-forward sublayer
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x


class GPTModel(nn.Module):
    """The full decoder-only language model.

    Token embeddings and learned positional embeddings are summed, passed
    through `n_layers` transformer blocks and a final normalization, then
    projected to one logit per vocabulary entry.

    The output head is *untied* from the input embedding: both are
    50257 x 768 = 38.6M parameters, and keeping them separate is what puts this
    model at 134M rather than the 96M a tied version would have.
    """

    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        # One positional vector per slot 0..seq_len-1, added to every sequence
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

    def param_summary(self):
        """Parameter counts, separating weights from the non-trainable masks."""
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        buffers = sum(b.numel() for b in self.buffers())
        return {"trainable": trainable, "buffers": buffers,
                "total": trainable + buffers}


def load_checkpoint(path, cfg, device="cpu"):
    """Build the model and load weights from a checkpoint file.

    Accepts both a bare state_dict (the format of the released checkpoint) and a
    dict with the weights under "model", so newer checkpoints saved by train.py
    load through the same function.
    """
    state = torch.load(path, map_location=device, weights_only=True)
    if isinstance(state, dict) and "model" in state:
        state = state["model"]
    model = GPTModel(cfg)
    model.load_state_dict(state)
    model.to(device).eval()
    return model


if __name__ == "__main__":
    from config import GPT_CONFIG_134M, describe

    model = GPTModel(GPT_CONFIG_134M)
    summary = model.param_summary()
    print(f"architecture : {describe()}")
    print(f"trainable    : {summary['trainable']:,} ({summary['trainable'] / 1e6:.2f}M)")
    print(f"mask buffers : {summary['buffers']:,}")
    print(f"total tensors: {summary['total']:,}")


In [ ]:
%%writefile data.py
"""Batch sampling from a memory-mapped token file.

The corpus is a flat array of uint16 token IDs on disk, several gigabytes of it.
`np.memmap` lets the operating system page in only the slices actually touched,
so training reads from an 8B-token file while using almost no RAM.

A batch is `batch_size` random windows of `block_size` tokens. Targets are the
same windows shifted one position right, which is the whole of the next-token
prediction objective: predict token i+1 from tokens 0..i.

Change from the original notebook, made deliberately (see docs/AUDIT.md):
`get_batch` takes `batch_size` and `block_size` as explicit arguments. In the
notebook they were module-level globals read at call time, and a later cell
reassigned both. The sampler silently switched from the configured 64 x 256 to
32 x 128 while every printed summary still reported 64 x 256. Passing them as
arguments makes that class of bug impossible, and
tests/test_model.py asserts the returned shape follows the arguments.
"""
import numpy as np
import torch


def load_tokens(path):
    """Open a uint16 token file without reading it into memory."""
    return np.memmap(path, dtype=np.uint16, mode="r")


def count_tokens(path):
    """How many tokens a .bin file holds."""
    return len(load_tokens(path))


def get_batch(path, batch_size, block_size, device):
    """Sample one batch of random windows from the token file at `path`.

    Returns (x, y), both int64 tensors of shape (batch_size, block_size), where
    y is x shifted forward by one token.

    The memmap is recreated on every call on purpose. Holding one open across
    thousands of iterations leaks memory, because the pages it touches are never
    released. This follows nanoGPT's approach.
    """
    data = np.memmap(path, dtype=np.uint16, mode="r")

    # Random start offsets. The -1 leaves room for the target window, which
    # extends one token further than the input window.
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))

    x = torch.stack([
        torch.from_numpy(data[i:i + block_size].astype(np.int64)) for i in ix])
    y = torch.stack([
        torch.from_numpy(data[i + 1:i + 1 + block_size].astype(np.int64)) for i in ix])

    if device.type == "cuda":
        # Pinned memory can be copied to the GPU asynchronously, so the transfer
        # overlaps with computation instead of blocking on it.
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)

    return x, y


if __name__ == "__main__":
    import sys

    path = sys.argv[1] if len(sys.argv) > 1 else "train.bin"
    n = count_tokens(path)
    print(f"{path}: {n:,} tokens ({n * 2 / 1e9:.2f} GB on disk as uint16)")

    x, y = get_batch(path, batch_size=2, block_size=8, device=torch.device("cpu"))
    print(f"x shape {tuple(x.shape)}, y shape {tuple(y.shape)}")
    print(f"x[0]: {x[0].tolist()}")
    print(f"y[0]: {y[0].tolist()}   <- x shifted by one")


In [ ]:
%%writefile tokenize_data.py
"""Turn a text dataset into a flat uint16 file of GPT-2 BPE token IDs.

This is the step that produced the 8B-token training corpus. The constraint was
a free Colab session: the full FineWeb-Edu snapshot is far larger than the
available disk and RAM, so the dataset is *streamed* rather than downloaded,
tokenized in chunks, and written straight into a pre-allocated memory-mapped
file. Peak RAM stays at roughly one chunk regardless of corpus size.

uint16 is deliberate: the GPT-2 vocabulary tops out at token ID 50256, which
fits in 16 bits. Storing IDs as int32 or int64 would double or quadruple a
16 GB file for no benefit.

Usage:
    # the 8B-token training corpus (long: this is the expensive step)
    python tokenize_data.py --out train.bin --max-tokens 8e9 --dump CC-MAIN-2024-10

    # a small held-out set from a DIFFERENT crawl, so it is disjoint from training
    python tokenize_data.py --out validation.bin --max-tokens 5e6 --dump CC-MAIN-2024-18

    # TinyStories instead of FineWeb-Edu (much smaller, useful for a quick run)
    python tokenize_data.py --dataset roneneldan/TinyStories --out train.bin --max-tokens 5e8
"""
import argparse
import os

import numpy as np
import tiktoken
from tqdm.auto import tqdm


def parse_args():
    p = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--dataset", default="HuggingFaceFW/fineweb-edu",
                   help="HuggingFace dataset id")
    p.add_argument("--dump", default="CC-MAIN-2024-10",
                   help="FineWeb-Edu crawl (dataset config). Ignored for other datasets. "
                        "Use a different crawl for validation so it is disjoint from training.")
    p.add_argument("--split", default="train")
    p.add_argument("--out", default="train.bin", help="output uint16 token file")
    p.add_argument("--max-tokens", type=float, default=8e9,
                   help="stop after writing this many tokens")
    p.add_argument("--min-doc-tokens", type=int, default=10,
                   help="skip documents shorter than this; they are mostly boilerplate")
    p.add_argument("--chunk-tokens", type=int, default=1_000_000,
                   help="buffer this many tokens before each write")
    return p.parse_args()


def main():
    args = parse_args()
    max_tokens = int(args.max_tokens)

    if os.path.exists(args.out):
        existing = np.memmap(args.out, dtype=np.uint16, mode="r")
        print(f"{args.out} already exists with {len(existing):,} tokens. "
              f"Delete it first to rebuild.")
        return

    from datasets import load_dataset

    # FineWeb-Edu needs a crawl name; most other datasets do not take one.
    name = args.dump if "fineweb" in args.dataset else None
    print(f"streaming {args.dataset}" + (f" [{name}]" if name else ""))
    ds = load_dataset(args.dataset, name=name, split=args.split, streaming=True)

    enc = tiktoken.get_encoding("gpt2")

    # Pre-allocate the full file, then truncate at the end. Growing a memmap
    # incrementally would mean repeated reallocation and copying.
    tmp = args.out + ".tmp"
    arr = np.memmap(tmp, dtype=np.uint16, mode="w+", shape=(max_tokens,))

    pos = 0
    buffer = []
    skipped = 0
    progress = tqdm(total=max_tokens, unit="tok", unit_scale=True, desc=args.out)

    for example in ds:
        # encode_ordinary ignores special tokens such as <|endoftext|>, so no
        # control token from the source text can leak into the corpus.
        ids = enc.encode_ordinary(example["text"])
        if len(ids) < args.min_doc_tokens:
            skipped += 1
            continue

        buffer.extend(ids)

        if len(buffer) >= args.chunk_tokens:
            take = min(len(buffer), max_tokens - pos)
            arr[pos:pos + take] = buffer[:take]
            pos += take
            progress.update(take)
            buffer = []
            if pos >= max_tokens:
                break

    # Flush whatever is left in the buffer
    if buffer and pos < max_tokens:
        take = min(len(buffer), max_tokens - pos)
        arr[pos:pos + take] = buffer[:take]
        pos += take
        progress.update(take)

    progress.close()
    arr.flush()
    del arr

    # Cut the file down to what was actually written. uint16 = 2 bytes per token.
    with open(tmp, "r+b") as f:
        f.truncate(pos * 2)
    os.rename(tmp, args.out)

    print(f"wrote {args.out}: {pos:,} tokens ({pos * 2 / 1e9:.2f} GB)")
    print(f"skipped {skipped:,} documents shorter than {args.min_doc_tokens} tokens")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
"""Pretraining loop: mixed precision, gradient accumulation, warmup + cosine decay.

The techniques here exist because of one constraint: a single free-tier 16 GB
notebook GPU.

  Mixed precision (fp16)   halves activation memory and uses the GPU's tensor
                           cores. A GradScaler multiplies the loss before
                           backward so small gradients do not underflow to zero
                           in fp16, then unscales before the optimizer step.
  Gradient accumulation    32 micro-batches of 32 sequences are accumulated
                           before one optimizer step, giving the gradient
                           quality of a 1024-sequence batch at the memory cost
                           of 32. This is the main reason the run fits in 16 GB.
  Gradient clipping        caps the global gradient norm at 1.0, which stops a
                           single bad batch from destabilizing training.
  Pinned async transfers   see data.py: host-to-device copies overlap with compute.

Three fixes relative to the original notebook, all documented in docs/AUDIT.md
and each covered by a test:

  1. One optimizer. The notebook built a scheduler around one optimizer and then
     passed a different, freshly constructed one to the training loop, so every
     scheduler.step() updated an object that no longer touched the model. Here
     the learning rate is computed by a plain function and written into the live
     optimizer, so there is no second object to fall out of sync with.
  2. The cosine floor is derived as lr/10 rather than typed separately. The
     notebook set min_lr=5e-4 against a peak of 1e-4, so its "decay" would have
     been a climb.
  3. Batch shape is passed explicitly instead of read from globals.

Every run writes its full configuration, the realized token count, throughput
and loss curves to runs/<name>/, because the original run recorded none of that
and the bugs above went unnoticed for months as a direct result.

Usage:
    python train.py --train-bin train.bin --val-bin validation.bin \
        --train-tokens 100e6 --lr 4e-4 --run-name baseline
"""
import argparse
import json
import math
import os
import random
import subprocess
import sys
import time
from contextlib import nullcontext

import numpy as np
import torch
from torch.amp import GradScaler, autocast

from config import GPT_CONFIG_134M, TRAIN_CONFIG
from data import get_batch
from model import GPTModel


def parse_args():
    p = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    # data
    p.add_argument("--train-bin", default="train.bin")
    p.add_argument("--val-bin", default="validation.bin")
    # architecture (defaults reproduce the released checkpoint)
    p.add_argument("--n-layers", type=int, default=GPT_CONFIG_134M["n_layers"])
    p.add_argument("--n-heads", type=int, default=GPT_CONFIG_134M["n_heads"])
    p.add_argument("--emb-dim", type=int, default=GPT_CONFIG_134M["emb_dim"])
    p.add_argument("--context", type=int, default=GPT_CONFIG_134M["context_length"])
    p.add_argument("--dropout", type=float, default=GPT_CONFIG_134M["drop_rate"])
    # optimization
    p.add_argument("--batch-size", type=int, default=TRAIN_CONFIG["batch_size"])
    p.add_argument("--grad-accum", type=int,
                   default=TRAIN_CONFIG["gradient_accumulation_steps"])
    p.add_argument("--lr", type=float, default=TRAIN_CONFIG["learning_rate"],
                   help="peak learning rate, reached at the end of warmup")
    p.add_argument("--min-lr", type=float, default=None,
                   help="cosine floor; defaults to lr/10 so it cannot exceed the peak")
    p.add_argument("--warmup-steps", type=int, default=200)
    p.add_argument("--weight-decay", type=float, default=TRAIN_CONFIG["weight_decay"])
    p.add_argument("--grad-clip", type=float, default=TRAIN_CONFIG["grad_clip"])
    p.add_argument("--train-tokens", type=float, default=100e6,
                   help="stop after this many tokens; a fixed budget makes runs comparable")
    # evaluation and bookkeeping
    p.add_argument("--eval-every", type=int, default=50, help="in optimizer steps")
    p.add_argument("--eval-iters", type=int, default=20, help="batches per loss estimate")
    p.add_argument("--seed", type=int, default=123)
    p.add_argument("--dtype", choices=["float16", "bfloat16", "float32"],
                   default="float16",
                   help="float16 on T4/V100/P100, bfloat16 on A100/H100, float32 on CPU")
    p.add_argument("--run-name", default="run")
    p.add_argument("--out-dir", default="runs")
    p.add_argument("--resume", default=None, help="path to a ckpt_last.pt to continue from")
    return p.parse_args()


def calc_loss_batch(input_batch, target_batch, model, vocab_size):
    """Cross-entropy between predicted logits and the next-token targets.

    Logits arrive as (batch, tokens, vocab) and targets as (batch, tokens).
    Both are flattened so every token position across the batch counts as one
    independent classification over the vocabulary.
    """
    logits = model(input_batch)
    return torch.nn.functional.cross_entropy(
        logits.view(-1, vocab_size), target_batch.view(-1))


@torch.no_grad()
def estimate_loss(model, path, args, device, ctx):
    """Average loss over several random batches.

    A single batch is far too noisy to compare checkpoints against, so this
    averages `eval_iters` of them. Dropout is disabled via model.eval() and
    turned back on afterwards, since evaluating with dropout active would report
    a loss the model does not actually have.
    """
    model.eval()
    losses = []
    for _ in range(args.eval_iters):
        x, y = get_batch(path, args.batch_size, args.context, device)
        with ctx:
            losses.append(calc_loss_batch(x, y, model, GPT_CONFIG_134M["vocab_size"]).item())
    model.train()
    return float(np.mean(losses))


def get_lr(step, peak_lr, min_lr, warmup_steps, max_steps):
    """Linear warmup, then cosine decay to the floor.

    Warmup: the parameters start random, so the first gradients are large and
    poorly directed. Ramping the learning rate from near zero avoids blowing up
    the weights before the model has learned anything.

    Cosine decay: large steps early to cover ground, progressively smaller steps
    later to settle into a minimum instead of bouncing around it.

    Returns a float. The caller writes it into the optimizer, which is what
    keeps schedule and optimizer from desynchronizing.
    """
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    progress = min(progress, 1.0)  # clamp so the LR never rises again past the end
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))


def git_commit():
    """Record which version of the code produced a run, or None outside git."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=os.path.dirname(os.path.abspath(__file__)),
            text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None


class RunLogger:
    """Writes everything needed to reconstruct and trust a run.

    config.json    every hyperparameter, the git commit, the seed, library
                   versions, the GPU name, and the exact command line
    metrics.jsonl  one line per evaluation
    summary.json   final losses, total tokens, throughput, peak memory
    loss_curve.png train and validation loss against tokens seen
    """

    def __init__(self, out_dir, run_name, config):
        self.dir = os.path.join(out_dir, run_name)
        os.makedirs(self.dir, exist_ok=True)
        self.t0 = time.time()
        self.history = []

        config = dict(config)
        config["git_commit"] = git_commit()
        config["command"] = " ".join(sys.argv)
        config["python_version"] = sys.version.split()[0]
        config["torch_version"] = torch.__version__
        config["device_name"] = (torch.cuda.get_device_name(0)
                                 if torch.cuda.is_available() else "cpu")
        self._write("config.json", config)
        print(f"[logger] writing to {self.dir}")

    def _write(self, name, obj):
        with open(os.path.join(self.dir, name), "w") as f:
            json.dump(obj, f, indent=2)

    def log(self, **metrics):
        metrics["wall_time_s"] = round(time.time() - self.t0, 1)
        self.history.append(metrics)
        with open(os.path.join(self.dir, "metrics.jsonl"), "a") as f:
            f.write(json.dumps(metrics) + "\n")

    def finish(self, **summary):
        summary["total_wall_time_s"] = round(time.time() - self.t0, 1)
        self._write("summary.json", summary)
        self._plot()
        print(f"[logger] run complete: {self.dir}")

    def _plot(self):
        if not self.history:
            return
        try:
            import matplotlib
            matplotlib.use("Agg")  # no display on a headless machine
            import matplotlib.pyplot as plt
        except ImportError:
            return
        tokens = [m["tokens_seen"] for m in self.history]
        plt.figure(figsize=(7, 4))
        plt.plot(tokens, [m["train_loss"] for m in self.history], label="train loss")
        plt.plot(tokens, [m["val_loss"] for m in self.history], label="val loss")
        plt.xlabel("tokens seen")
        plt.ylabel("cross-entropy loss")
        plt.title(os.path.basename(self.dir))
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dir, "loss_curve.png"), dpi=120)
        plt.close()


def main():
    args = parse_args()

    # Seed everything so a rerun with the same flags gives the same run. Without
    # this, two "identical" ablation runs differ by more than the variable under test.
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16,
             "float32": torch.float32}[args.dtype]
    use_amp = device.type == "cuda" and dtype != torch.float32
    ctx = autocast(device_type="cuda", dtype=dtype) if use_amp else nullcontext()
    # Loss scaling is only needed for fp16. bfloat16 has fp32's exponent range,
    # so gradients cannot underflow the same way.
    scaler = GradScaler(enabled=use_amp and dtype == torch.float16)

    model_cfg = {
        "vocab_size": GPT_CONFIG_134M["vocab_size"],
        "context_length": args.context,
        "emb_dim": args.emb_dim,
        "n_heads": args.n_heads,
        "n_layers": args.n_layers,
        "drop_rate": args.dropout,
        "qkv_bias": GPT_CONFIG_134M["qkv_bias"],
    }
    if args.min_lr is None:
        args.min_lr = args.lr / 10  # derived, so it can never exceed the peak

    tokens_per_step = args.batch_size * args.context * args.grad_accum
    max_steps = max(1, int(args.train_tokens) // tokens_per_step)

    logger = RunLogger(args.out_dir, args.run_name, {
        "args": vars(args),
        "model_config": model_cfg,
        "tokens_per_optimizer_step": tokens_per_step,
        "max_optimizer_steps": max_steps,
    })

    model = GPTModel(model_cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{n_params / 1e6:.2f}M parameters on {device}")
    print(f"{tokens_per_step:,} tokens/step x {max_steps:,} steps "
          f"= {tokens_per_step * max_steps / 1e6:.1f}M tokens")

    # betas=(0.9, 0.95) rather than the default 0.999: the shorter second-moment
    # window adapts faster, which is standard for language model pretraining.
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=args.lr, betas=(0.9, 0.95),
        weight_decay=args.weight_decay, eps=1e-9)

    step, tokens_seen = 0, 0
    if args.resume:
        ckpt = torch.load(args.resume, map_location=device, weights_only=True)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        step, tokens_seen = ckpt["step"], ckpt["tokens_seen"]
        print(f"resumed from {args.resume}: step {step}, {tokens_seen:,} tokens")

    def save(name):
        torch.save({"model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "step": step, "tokens_seen": tokens_seen,
                    "model_config": model_cfg, "args": vars(args)},
                   os.path.join(logger.dir, name))

    model.train()
    optimizer.zero_grad(set_to_none=True)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    best_val = float("inf")
    window_tokens, window_t0, train_start = 0, time.time(), time.time()

    while step < max_steps:
        lr = get_lr(step, args.lr, args.min_lr, args.warmup_steps, max_steps)
        for group in optimizer.param_groups:
            group["lr"] = lr  # the schedule reaches the model through this line

        for _ in range(args.grad_accum):
            x, y = get_batch(args.train_bin, args.batch_size, args.context, device)
            with ctx:
                loss = calc_loss_batch(x, y, model, model_cfg["vocab_size"])
                # Divide by grad_accum so the accumulated gradient is the mean
                # over all micro-batches rather than their sum.
                loss = loss / args.grad_accum
            scaler.scale(loss).backward()
            # Count tokens from the tensor that actually went through the model,
            # never from the config. This is what the original run got wrong.
            tokens_seen += x.numel()
            window_tokens += x.numel()

        scaler.unscale_(optimizer)  # clip real gradients, not scaled ones
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % args.eval_every == 0 or step == max_steps:
            train_loss = estimate_loss(model, args.train_bin, args, device, ctx)
            val_loss = estimate_loss(model, args.val_bin, args, device, ctx)
            tok_per_s = window_tokens / max(time.time() - window_t0, 1e-9)
            peak_mem = (torch.cuda.max_memory_allocated() / 1e9
                        if device.type == "cuda" else 0.0)

            logger.log(step=step, tokens_seen=tokens_seen, lr=lr,
                       train_loss=round(train_loss, 4), val_loss=round(val_loss, 4),
                       val_perplexity=round(math.exp(val_loss), 2),
                       tokens_per_sec=round(tok_per_s),
                       peak_gpu_mem_gb=round(peak_mem, 2))
            print(f"step {step:5d}/{max_steps} | {tokens_seen / 1e6:7.1f}M tok | "
                  f"train {train_loss:.3f} | val {val_loss:.3f} | "
                  f"ppl {math.exp(val_loss):7.2f} | lr {lr:.2e} | "
                  f"{tok_per_s / 1e3:.1f}K tok/s | {peak_mem:.1f} GB")

            save("ckpt_last.pt")
            if val_loss < best_val:
                best_val = val_loss
                save("ckpt_best.pt")
            window_tokens, window_t0 = 0, time.time()

    total_time = time.time() - train_start
    final_val = estimate_loss(model, args.val_bin, args, device, ctx)
    logger.finish(
        final_val_loss=round(final_val, 4),
        final_val_perplexity=round(math.exp(final_val), 2),
        best_val_loss=round(best_val, 4),
        tokens_seen=tokens_seen,
        optimizer_steps=step,
        avg_tokens_per_sec=round(tokens_seen / max(total_time, 1e-9)),
        peak_gpu_mem_gb=(round(torch.cuda.max_memory_allocated() / 1e9, 2)
                         if device.type == "cuda" else 0.0),
        n_params=n_params,
    )


if __name__ == "__main__":
    main()


## 5. Build a small corpus

Throughput depends on tensor shapes and hardware, not on what the tokens say,
so 50M tokens measures the same number as the full 8B corpus and takes minutes
rather than hours.

Validation comes from a **different Common Crawl snapshot** than training, so
it is distribution-matched but disjoint. Same split discipline as the real run.


In [ ]:
!python tokenize_data.py --out train.bin      --max-tokens 50e6 --dump CC-MAIN-2024-10
!python tokenize_data.py --out validation.bin --max-tokens 2e6  --dump CC-MAIN-2024-18


In [ ]:
import numpy as np
for f in ('train.bin', 'validation.bin'):
    n = len(np.memmap(f, dtype=np.uint16, mode='r'))
    print(f'{f}: {n:,} tokens ({n * 2 / 1e9:.2f} GB on disk)')


## 6. Run the measurement

20M tokens is about 150 optimizer steps, well past where throughput settles.

`--dtype float16` is right for both the P100 and the T4: both run fp16 at full
rate, and neither supports bfloat16. `torch.compile` is not used, since its
Triton backend needs sm_70+ and would exclude the P100 entirely.

Ignore the first throughput figure printed; it includes CUDA context creation.


In [ ]:
!python train.py \
    --train-bin train.bin --val-bin validation.bin \
    --batch-size 32 --context 128 --grad-accum 32 \
    --lr 4e-4 --dtype float16 \
    --train-tokens 20e6 --eval-every 10 --run-name gpu_throughput


## 7. The two numbers

Report throughput **with the hardware, dtype and batch shape attached**. The
same code on the same GPU varies by more than 2x between small-batch fp32 and
large-batch fp16, so a bare tokens/sec figure is not checkable. That is exactly
why the unsupported "75K tokens/sec" was removed from the CV.


In [ ]:
import json, os

run = 'runs/gpu_throughput'
if not os.path.exists(f'{run}/summary.json'):
    raise SystemExit('No run found. Section 6 did not finish; check its output above.')

summary = json.load(open(f'{run}/summary.json'))
config  = json.load(open(f'{run}/config.json'))

print('=' * 66)
print('GPU              :', config['device_name'])
print('torch            :', config['torch_version'])
print('dtype            :', config['args']['dtype'])
print('batch x context  :', config['args']['batch_size'], 'x', config['args']['context'])
print('grad accumulation:', config['args']['grad_accum'])
print('tokens per step  :', f"{config['tokens_per_optimizer_step']:,}")
print('-' * 66)
print('AVG TOKENS/SEC   :', f"{summary['avg_tokens_per_sec']:,}")
print('PEAK GPU MEMORY  :', summary['peak_gpu_mem_gb'], 'GB   (of 16 GB available)')
print('-' * 66)
print('tokens seen      :', f"{summary['tokens_seen']:,}")
print('optimizer steps  :', summary['optimizer_steps'])
print('wall time        :', summary['total_wall_time_s'], 's')
print('final val loss   :', summary['final_val_loss'])
print('final val ppl    :', summary['final_val_perplexity'])
print('=' * 66)
print()
print('COPY THIS LINE:')
print(f"  {summary['avg_tokens_per_sec']:,} tokens/sec, "
      f"{summary['peak_gpu_mem_gb']} GB peak, on one {config['device_name']} "
      f"at batch {config['args']['batch_size']} x {config['args']['context']}, "
      f"{config['args']['dtype']}, grad accum {config['args']['grad_accum']}, "
      f"torch {config['torch_version']}")


### Throughput settling, and the loss curve

The first row is inflated by startup cost. The rest should be roughly flat,
which is what makes the average meaningful.


In [ ]:
import json

print(f"{'step':>6} {'tok/s':>9} {'peak GB':>9} {'train':>8} {'val':>8}")
for line in open('runs/gpu_throughput/metrics.jsonl'):
    m = json.loads(line)
    print(f"{m['step']:>6} {m['tokens_per_sec']:>9,} "
          f"{m['peak_gpu_mem_gb']:>9.2f} {m['train_loss']:>8.3f} {m['val_loss']:>8.3f}")


In [ ]:
import os
from IPython.display import Image, display

curve = 'runs/gpu_throughput/loss_curve.png'
if os.path.exists(curve):
    display(Image(curve))
else:
    print('No loss curve (matplotlib unavailable). The numbers above are unaffected.')


## 8. Verify the embedded code, and keep the log

These hashes should match the table in the project's `docs/MEASURE.md`. That
is what makes this notebook's numbers attributable to the repository's code
rather than to something pasted into a cell.

Then download the archive and commit `runs/` to the repo, so the published
figure and the log that produced it travel together. Checkpoints are
gitignored; only logs and the curve are kept.


In [ ]:
import hashlib

for f in ['config.py', 'model.py', 'data.py', 'tokenize_data.py', 'train.py']:
    h = hashlib.sha256(open(f, encoding='utf-8').read().encode()).hexdigest()[:16]
    print(f'{f:20s} sha256:{h}')


In [ ]:
!tar -czf /kaggle/working/gpu_throughput_run.tar.gz runs/gpu_throughput
!ls -lh /kaggle/working/gpu_throughput_run.tar.gz
print()
print('Download it from the Output panel on the right.')
